# TXPipe - Cluster shear calibrator

Test notebook for the `CLClusterShearCalibrator` stage (`txpipe/extensions/cluster_counts/cluster_shear_calibrator.py`), which computes the shear response of the sources of each cluster stack, and for its use in `CLClusterEnsembleProfiles` with `shear_calibration="cluster"`.

The upstream stages of `tangential_shear_selection.ipynb` (source selection, cluster-source pairs, cluster binning) are run once and their outputs are reused. After that, only the calibrator stage is run, and it is reloaded every time so edits to the stage are picked up without restarting the kernel.

In [ ]:
import os
import sys
import importlib
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import h5py
import ceci
import pickle

In [ ]:
my_txpipe_dir = "/sps/lsst/users/ebarroso/TXPipe"
sys.path.append(my_txpipe_dir)
sys.path.append(f"{my_txpipe_dir}/txpipe/extensions")
import txpipe
import txpipe.extensions as extensions

## Files

All the outputs of this notebook go to `outputs_cluster_calibration`, so the other notebooks are not affected. Set `RERUN_UPSTREAM = True` to regenerate the upstream outputs.

In [ ]:
inputs_dir = f"{my_txpipe_dir}/data/example/inputs"
out_dir = f"{my_txpipe_dir}/data/example/outputs_cluster_calibration"
os.makedirs(out_dir, exist_ok=True)

shear_catalog = f"{inputs_dir}/metadetect_shear_catalog.hdf5"
cluster_catalog = f"{inputs_dir}/cluster_catalog.hdf5"
photoz_pdfs = f"{inputs_dir}/photoz_pdfs.hdf5"
fiducial_cosmology = f"{my_txpipe_dir}/data/fiducial_cosmology.yml"
shear_selection = f"{out_dir}/shear_selection.hdf5"
shear_bin_assignments = f"{out_dir}/shear_bin_assignments.hdf5"
shear_tomography_catalog = f"{out_dir}/shear_tomography_catalog.hdf5"

cluster_shear_selection = f"{out_dir}/cluster_shear_selection.hdf5"
cluster_catalog_tomography = f"{out_dir}/cluster_catalog_tomography.hdf5"
cluster_shear_response = f"{out_dir}/cluster_shear_response.hdf5"
cluster_shear_response_stack = f"{out_dir}/cluster_shear_response_stack.hdf5"

RERUN_UPSTREAM = False

---

## Upstream stages (run once)

Same stages and options as `tangential_shear_selection.ipynb`, with one configuration only: $\Delta\Sigma$ with zmode for the cut and the weights.

In [ ]:
if RERUN_UPSTREAM or not os.path.exists(shear_tomography_catalog):
    step1 = txpipe.TXMetaDetectSelection.make_stage(
        shear_catalog=shear_catalog,
        shear_selection=shear_selection,
        output_dir=out_dir,
    )
    step1.run()
    step1.finalize()

    # tomographic bins from the true redshifts
    step1b = txpipe.TXSimpleRedshiftSourceTomography.make_stage(
        shear_catalog=shear_catalog,
        shear_bin_assignments=shear_bin_assignments,
        output_dir=out_dir,
        source_zbin_edges=[0.5, 0.7, 0.9, 1.1, 2.0],
        chunk_rows = 100_000,
    )
    step1b.run()
    step1b.finalize()

    step1c = txpipe.TXComputeShearCalibrationMetadetect.make_stage(
        shear_catalog=shear_catalog,
        shear_bin_assignments=shear_bin_assignments,
        shear_selection=shear_selection,
        shear_tomography_catalog=shear_tomography_catalog,
        output_dir=out_dir,
        delta_gamma=0.02,
    )
    step1c.run()
    step1c.finalize()

In [ ]:
if RERUN_UPSTREAM or not os.path.exists(cluster_shear_selection):
    step2 = extensions.CLClusterSourceSelect.make_stage(
        shear_catalog=shear_catalog,
        cluster_catalog=cluster_catalog,
        fiducial_cosmology=fiducial_cosmology,
        shear_selection=shear_selection,
        shear_tomography_catalog=shear_tomography_catalog,
        source_photoz_pdfs=photoz_pdfs,
        cluster_shear_selection=cluster_shear_selection,
        output_dir=out_dir,
        max_radius=10.0,
        use_radius=True,
        max_angle=40,
        redshift_cut_criterion="zmode",
    )
    step2.run()
    step2.finalize()

In [ ]:
if RERUN_UPSTREAM or not os.path.exists(cluster_catalog_tomography):
    step3 = extensions.CLClusterBinningRedshiftRichness.make_stage(
        cluster_catalog=cluster_catalog,
        zedge=[0.2, 0.4, 0.6, 0.8],
        richedge=[5, 10, 100],
        cluster_catalog_tomography=cluster_catalog_tomography,
    )
    step3.run()
    step3.finalize()

---

## Calibrator stage

Rerun from here after every change to `cluster_shear_calibrator.py`. `allow_reload` lets ceci register the reloaded class again under the same name.

In [ ]:
ceci.PipelineStage.allow_reload = True
from txpipe.extensions.cluster_counts import cluster_shear_calibrator
importlib.reload(cluster_shear_calibrator)
CLClusterShearCalibrator = cluster_shear_calibrator.CLClusterShearCalibrator
print(CLClusterShearCalibrator.config_options)

In [ ]:
calib_args = dict(
    cluster_catalog_tomography=cluster_catalog_tomography,
    cluster_shear_selection=cluster_shear_selection,
    fiducial_cosmology=fiducial_cosmology,
    shear_catalog=shear_catalog,
    shear_tomography_catalog=shear_tomography_catalog,
    shear_bin_assignments=shear_bin_assignments,
    shear_selection=shear_selection,
    output_dir=out_dir,
    r_min=1.3,
    r_max=3.0,
    nbins=3,
    bin_units="mpc",
)

# One response per radial bin of each stack
calib = CLClusterShearCalibrator.make_stage(
    cluster_shear_response=cluster_shear_response,
    radial_calibration=True,
    **calib_args,
)
calib.run()
calib.finalize()

# One response per stack, to compare
calib_stack = CLClusterShearCalibrator.make_stage(
    cluster_shear_response=cluster_shear_response_stack,
    radial_calibration=False,
    **calib_args,
)
calib_stack.run()
calib_stack.finalize()

## Clusters and sources of one richness-redshift bin

In [ ]:
with h5py.File(cluster_catalog_tomography, "r") as f:
    bin_names = list(f["cluster_bin"].keys())
print(bin_names)

bin_name = bin_names[0]
clusters, sources = calib.load_cluster_bin_sources(bin_name)
print(clusters.meta)
clusters

In [ ]:
print(sources.meta)
sources[:10]

Number of sources per cluster and radial bin. Sources with `radial_bin = -1` are outside `[r_min, r_max]`.

In [ ]:
nbins = len(calib.distance_bins) - 1
in_range = sources["radial_bin"] >= 0
print("Sources outside the radial range:", np.sum(~in_range))
print("Sources per radial bin:", np.bincount(sources["radial_bin"][in_range], minlength=nbins))

counts = np.zeros((len(clusters), nbins), dtype=int)
for i, cid in enumerate(clusters["cluster_id"]):
    sel = in_range & (sources["cluster_id"] == cid)
    counts[i] = np.bincount(sources["radial_bin"][sel], minlength=nbins)
for cid, c in zip(clusters["cluster_id"], counts):
    print(cid, c)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for b in range(nbins):
    sel = sources["radial_bin"] == b
    ax.hist(sources["distance"][sel], bins=50, range=(0, sources["distance"].max()), histtype="step", label=f"bin {b}")
ax.hist(sources["distance"][~in_range], bins=50, range=(0, sources["distance"].max()), color="gray", alpha=0.3, label="outside")
for edge in calib.distance_bins:
    ax.axvline(edge, color="k", lw=0.5)
ax.set_xlabel(f"Distance [{sources.meta['bin_units']}]")
ax.set_ylabel("Sources")
ax.legend()
plt.tight_layout()
plt.show()

---

## Response

The stage saves, for each stack, the full response matrix R of each radial bin (`radial_calibration=True`) or of the whole stack (`radial_calibration=False`), which is what the ensemble applies, and its bootstrap error `R_err`. Below are its diagonal elements. On this example catalog the true response is 1 for every object and there is no magnification, so there is no radial trend to find: R is dominated by selection noise, because `psf_T_mean` is drawn independently in each metadetect variant. This only checks that the stage runs.

In [ ]:
with h5py.File(shear_tomography_catalog, "r") as f:
    R_global = f["response/R_2d"][:]
print("global R_2d:", np.round(R_global.ravel(), 4))

with h5py.File(cluster_shear_response, "r") as fr, h5py.File(cluster_shear_response_stack, "r") as fs:
    print(dict(fr["calibration"].attrs))
    edges = fr["calibration"].attrs["distance_bins"]
    units = fr["calibration"].attrs["bin_units"]
    stacks = [name for name in fr["calibration"] if "R" in fr["calibration"][name]]
    R_radial = {name: fr["calibration"][name]["R"][:] for name in stacks}
    err_radial = {name: fr["calibration"][name]["R_err"][:] for name in stacks}
    R_stack = {name: fs["calibration"][name]["R"][:] for name in stacks}
    n_radial = {name: fr["calibration"][name].attrs["n_sources_per_radial_bin"] for name in stacks}

for name in stacks:
    print(f"\n{name}: stack R = {np.round(R_stack[name].ravel(), 4)}")
    for k, n in enumerate(n_radial[name]):
        print(f"  radial bin {k} [{edges[k]:.2f}, {edges[k+1]:.2f}] {units}: n = {n}, "
              f"R = {np.round(R_radial[name][k].ravel(), 4)} +- {np.round(err_radial[name][k].ravel(), 4)}")

# R11 and R22 per radial bin (points) and per stack (dashed), global in black
r_mid = np.sqrt(edges[1:] * edges[:-1])
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for i, ax in enumerate(axes):
    for name in stacks:
        line, = ax.plot(r_mid, R_radial[name][:, i, i], "o-", label=name)
        ax.axhline(R_stack[name][i, i], color=line.get_color(), ls="--", lw=0.8)
    ax.axhline(R_global[i, i], color="k", lw=1)
    ax.set_xscale("log")
    ax.set_xlabel(f"Distance [{units}]")
    ax.set_ylabel(f"$R_{{{i + 1}{i + 1}}}$")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

---

## Profiles calibrated with the cluster response

`CLClusterEnsembleProfiles` with `shear_calibration="global"` calibrates every source with the R of the whole source sample (`shear_tomography_catalog`). With `shear_calibration="cluster"` it reads `cluster_shear_response` and calibrates the sources of each stack with its R, or with the R of their radial bin. The radial bins must be those of the calibrator, otherwise the stage stops with an error. Without the calibrator stage, `cluster_shear_response` is `"none"`.

In [ ]:
ensemble_args = dict(
    shear_catalog=shear_catalog,
    shear_tomography_catalog=shear_tomography_catalog,
    fiducial_cosmology=fiducial_cosmology,
    cluster_shear_selection=cluster_shear_selection,
    cluster_catalog_tomography=cluster_catalog_tomography,
    source_photoz_pdfs=photoz_pdfs,
    output_dir=out_dir,
    delta_sigma=True,
    redshift_weight_criterion="zmode",
    cov_type="sample_covariance",
    r_min=calib_args["r_min"],
    r_max=calib_args["r_max"],
    nbins=calib_args["nbins"],
    bin_units=calib_args["bin_units"],
)

runs = {
    "global": dict(shear_calibration="global", cluster_shear_response="none"),
    "stack": dict(shear_calibration="cluster", cluster_shear_response=cluster_shear_response_stack),
    "radial": dict(shear_calibration="cluster", cluster_shear_response=cluster_shear_response),
}
profiles = {}
for name, run_args in runs.items():
    ens = extensions.CLClusterEnsembleProfiles.make_stage(
        cluster_profiles=f"{out_dir}/cluster_profiles_{name}.pkl",
        cluster_source_quantities=f"{out_dir}/cluster_source_quantities_{name}.hdf5",
        **run_args,
        **ensemble_args,
    )
    ens.run()
    ens.finalize()
    with open(f"{out_dir}/cluster_profiles_{name}.pkl", "rb") as f:
        profiles[name] = pickle.load(f)

In [ ]:
stacks = [k for k, v in profiles["global"].items() if v["clmm_cluster_ensemble"] is not None]
fig, axes = plt.subplots(1, len(stacks), figsize=(4 * len(stacks), 3.5), squeeze=False)
for ax, k in zip(axes[0], stacks):
    for name, marker in zip(profiles, ["o", "s", "^"]):
        ens = profiles[name][k]["clmm_cluster_ensemble"]
        r = np.asarray(ens.stacked_data["radius"])
        err = np.sqrt(np.diag(ens.cov["tan_sc"]))
        ax.errorbar(r, ens.stacked_data["tangential_comp"], err, fmt=marker, ms=4, capsize=2, label=name)
    ax.set_xscale("log")
    ax.set_title(f"{k} ({profiles['global'][k]['n_cl']} clusters)", fontsize=8)
    ax.set_xlabel("R [Mpc]")
axes[0, 0].set_ylabel(r"$\Delta\Sigma$")
axes[0, 0].legend(fontsize=7)
plt.tight_layout()
plt.show()